In [2]:
from torch.utils.data import Dataset, DataLoader
import torch
from pathlib import Path
import numpy as np
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


In [3]:

sequences = torch.tensor([[1, 2, 0, 0], [5, 6, 7, 8]])
packed = pack_padded_sequence(sequences, lengths=(2, 4),
                              enforce_sorted=False, batch_first=True)
print(packed)
padded, lengths = pad_packed_sequence(packed, batch_first=True)
print(padded)


PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))
tensor([[1, 2, 0, 0],
        [5, 6, 7, 8]])


In [ ]:


class QuickDrawDataset(Dataset):
    def __init__(self, quickdraw_folder="quickdraw_data", split='train', max_per_class=10000):
        self.data_folder = Path(quickdraw_folder)
        # Exclude .full.npz files to avoid duplicates
        all_files = self.data_folder.glob("*.npz")
        self.files = sorted([f for f in all_files if not f.stem.endswith('.full')])
        self.split = split
        self.max_per_class = max_per_class
        self.label_to_idx = {f.stem: i for i, f in enumerate(self.files)}
        
        # Load all data at initialization
        self.data = []
        self.labels = []
        num_files_loaded = 0
        for file_path in self.files:
            label = file_path.stem
            label_idx = self.label_to_idx[label]
            
            # Load all drawings from this file
            data = np.load(file_path, allow_pickle=True, encoding='latin1')
            drawings = data[split]
            num_drawings = min(len(drawings), self.max_per_class)
            
            # Process each drawing
            for drawing_idx in range(num_drawings):
                drawing = drawings[drawing_idx]
                X = torch.tensor(drawing, dtype=torch.float32)
                
                # Pad or truncate to 50 rows
                if X.shape[0] >= 50:
                    X = X[:50]
                else:
                    padding_needed = 50 - X.shape[0]
                    padding = torch.zeros(padding_needed, 3)
                    X = torch.cat([X, padding], dim=0)
                
                self.data.append(X)
                self.labels.append(label_idx)
                num_files_loaded += 1
                print(f"[INFO] {num_files_loaded} / {len(self.label_to_idx)} files laded.")
                
            
            data.close()
        
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [23]:
dataset = QuickDrawDataset(split='train', max_per_class=100)
dataloader = DataLoader(dataset=dataset, batch_size=32, shuffle=True)
print(dataset.__getitem__(0)[0].shape)


Dataset initialized with 200 samples from 345 classes
torch.Size([50, 3])


In [27]:
X, y = next(iter(dataloader))
print(X.shape)
print(y.shape)

torch.Size([32, 50, 3])
torch.Size([32])


In [23]:
print(dataset.__getitem__(1))

(tensor([[  59.,   15.,    0.],
        [ 165.,   25.,    0.],
        [ 200.,   -4.,    0.],
        [  52.,   -8.,    0.],
        [  51.,   -1.,    0.],
        [  10.,    9.,    0.],
        [  10.,   18.,    0.],
        [ -21.,   17.,    0.],
        [ -46.,   27.,    0.],
        [ -33.,   26.,    0.],
        [ -49.,   22.,    0.],
        [ -86.,   17.,    0.],
        [ -28.,   -1.,    0.],
        [ -55.,   -9.,    0.],
        [ -99.,   -1.,    0.],
        [ -46.,   -7.,    0.],
        [ -68.,  -19.,    0.],
        [ -21.,  -36.,    0.],
        [  -1.,  -33.,    0.],
        [   8.,  -22.,    0.],
        [  -2.,  -41.,    1.],
        [ 408.,   31.,    0.],
        [  31.,  -65.,    0.],
        [   5.,  -27.,    0.],
        [  -1.,  -54.,    0.],
        [ -16.,  -25.,    0.],
        [   3.,  -33.,    0.],
        [  -7.,  -10.,    0.],
        [  62.,    1.,    1.],
        [   0.,   -4.,    0.],
        [  48.,   18.,    0.],
        [   0.,  115.,    0.],
       